In [1]:
import pandas as pd

In [2]:
data_pib = pd.read_csv("../data/Brutos/5_BANCO MUNDIAL_PIB PER CAPITA.csv", skiprows=4)
print(data_pib.shape)

(266, 71)


# 1. Me familiarizo con la estructura del dataset y observo su shape, info, head... 

In [3]:
data_pib.info()

<class 'pandas.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 71 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    266 non-null    str    
 1   Country Code    266 non-null    str    
 2   Indicator Name  266 non-null    str    
 3   Indicator Code  266 non-null    str    
 4   1960            152 non-null    float64
 5   1961            155 non-null    float64
 6   1962            157 non-null    float64
 7   1963            157 non-null    float64
 8   1964            157 non-null    float64
 9   1965            163 non-null    float64
 10  1966            164 non-null    float64
 11  1967            167 non-null    float64
 12  1968            168 non-null    float64
 13  1969            168 non-null    float64
 14  1970            190 non-null    float64
 15  1971            191 non-null    float64
 16  1972            191 non-null    float64
 17  1973            191 non-null    float64
 18  1

In [4]:
print(data_pib["Indicator Name"].unique())
data_pib.head(3)

<StringArray>
['GDP per capita (current US$)']
Length: 1, dtype: str


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,28440.041688,30082.158423,30654.485124,22664.370995,26827.344787,31000.571380,34897.618393,38590.565029,NaN,NaN
1,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,186.089515,186.909365,197.367876,225.400456,208.963066,226.836513,...,1529.923080,1553.617944,1508.031525,1351.503167,1560.894626,1675.902524,1571.132704,1628.227289,1722.38562,NaN
2,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,525.469771,491.337221,496.602504,510.787063,356.496214,357.261153,413.757895,416.871146,NaN,NaN


# 2 Creo la lista anios con un bucle en una sola línea: a va tomando cada número de 2015 a 2025, lo convierto en texto con str(a) y lo voy guardando en la lista (["2015", "2016", ..., "2024"]). Lo paso a texto porque así se llaman las columnas de años en el CSV del Banco Mundial, y esta lista me sirve para quedarme solo con esas columnas.

In [5]:
anios = [str(a) for a in range(2015, 2025)]
data_pib = data_pib[["Country Name", "Country Code"] + anios]
print(data_pib.shape)

(266, 12)


# 3 · Renombro columnas para dar homogeneidad entre diferentes datasets

In [6]:
data_pib = data_pib.rename(columns={"Country Name": "pais", "Country Code": "iso3"})

- `rename(columns={viejo: nuevo})` → mismos nombres que en el resto de datasets (la clave del merge es `iso3`).

# 4 · Quito categorías agregadas que no aportan nada :Mundo, UE, grupos de renta…

In [7]:
equivalencias = pd.read_csv("../data/Brutos/4_EQUIVALENCIAS_PAISES_ISO_3.csv")
data_pib = data_pib[data_pib["iso3"].isin(equivalencias["iso3"])]
print(data_pib.shape)

(215, 12)


- Leemos tu tabla de países reales.
- `isin(...)` → `True` si el código está en la tabla; `df[máscara]` deja solo esas filas.

**Debe salir:** `(215, 28)`.

# 5. Modifico formato: de ancho a largo con `melt`

In [8]:
data_pib = data_pib.melt(
    id_vars=["pais", "iso3"],
    var_name="anio",
    value_name="pib_pc_usd"
)
print(data_pib.shape)
print(data_pib)

(2150, 4)
              pais iso3  anio    pib_pc_usd
0            Aruba  ABW  2015  27458.220154
1      Afghanistan  AFG  2015    565.569730
2           Angola  AGO  2015   3641.728939
3          Albania  ALB  2015   4199.539129
4          Andorra  AND  2015  38654.934722
...            ...  ...   ...           ...
2145         Samoa  WSM  2024   5392.877619
2146   Yemen, Rep.  YEM  2024           NaN
2147  South Africa  ZAF  2024   6267.186814
2148        Zambia  ZMB  2024   1187.109434
2149      Zimbabwe  ZWE  2024   2496.155150

[2150 rows x 4 columns]


- `id_vars` → columnas fijas; `var_name` → nombre de la columna que recoge los años; `value_name` → la que recoge los valores.

**Debe salir:** `(5590, 4)` = 215 × 26.

# 6 · `anio` de string a entero

In [9]:
data_pib["anio"] = data_pib["anio"].astype(int)
print(data_pib.dtypes)

pais              str
iso3              str
anio            int64
pib_pc_usd    float64
dtype: object


- Tras `melt` el año es texto; con texto el merge con turismo/vivienda no casaría.

**Debe salir:** `anio` `int64`, `pib_pc_usd` `float64`.

# 7 · Ordenar columnas y reiniciar el index

In [10]:
data_pib = data_pib.sort_values(["pais", "anio"]).reset_index(drop=True)
data_pib.head()

,pais,iso3,anio,pib_pc_usd
0,Afghanistan,AFG,2015,565.569730
1,Afghanistan,AFG,2016,522.082216
2,Afghanistan,AFG,2017,525.469771
3,Afghanistan,AFG,2018,491.337221
4,Afghanistan,AFG,2019,496.602504


# 8 · Calcular el crecimiento % anual por país y darle nombre a la coumna ("crec. pib (%))

In [11]:
data_pib["crecim_pib_(%)"] = data_pib.groupby("iso3")["pib_pc_usd"].pct_change(fill_method=None) * 100
data_pib[data_pib["iso3"] == "ESP"].head(4)

,pais,iso3,anio,pib_pc_usd,crecim_pib_(%)
1770,Spain,ESP,2015,25982.441620,NaN
1771,Spain,ESP,2016,26755.606115,2.975719
1772,Spain,ESP,2017,28381.342560,6.076246
1773,Spain,ESP,2018,30602.421288,7.825841


- `groupby("iso3")` → trabaja país a país. **Clave:** sin esto, la primera fila de España se compararía con la última de Albania.
- `["pib_pc_usd"].pct_change()` → `(valor_actual − valor_anterior) / valor_anterior`. Ejemplo: 100 → 110 da `0.10`.
- `fill_method=None` → si falta un dato, no lo rellena con el anterior (no inventamos).
- `* 100` → de proporción a porcentaje (0.10 → 10.0), igual que inflación y vivienda.
- La primera fila de cada país queda en `NaN`: no tiene año anterior.
- Es crecimiento **nominal en dólares corrientes**, y por eso se ve afectado por el tipo de cambio. Anótalo en la memoria como limitación.

# 9 Dejo solo dos decimales

In [12]:
data_pib["crecim_pib_(%)"] = data_pib["crecim_pib_(%)"].round(1)
data_pib

,pais,iso3,anio,pib_pc_usd,crecim_pib_(%)
0,Afghanistan,AFG,2015,565.569730,NaN
1,Afghanistan,AFG,2016,522.082216,-7.7
2,Afghanistan,AFG,2017,525.469771,0.6
3,Afghanistan,AFG,2018,491.337221,-6.5
4,Afghanistan,AFG,2019,496.602504,1.1
...,...,...,...,...,...
2145,Zimbabwe,ZWE,2020,2059.637040,-5.7
2146,Zimbabwe,ZWE,2021,2613.616741,26.9
2147,Zimbabwe,ZWE,2022,2536.401971,-3.0
2148,Zimbabwe,ZWE,2023,2195.225020,-13.5


# 10 · Quito los NaN (sobre todo derivan de la comparativa con 2015, porque la serie histórica la he delimitado del 2015 al 2025, como no hay año anterior con el que calcular el crecimiento (2014), aparece Nan. Es una pérdida inevitable y afecta a los 215 países por igual) 

In [13]:
print(data_pib.isnull().sum())
data_pib = data_pib.dropna(subset=["crecim_pib_(%)"]).reset_index(drop=True)
print(data_pib.shape)

pais                0
iso3                0
anio                0
pib_pc_usd         80
crecim_pib_(%)    292
dtype: int64
(1858, 5)


- `isnull().sum()` → huecos por columna. **Debe salir:** `pib_pc_usd` 203 y `crecimiento_pib_pc_pct` 418 (203 sin dato + 215 primeras filas de cada país).
- `dropna(subset=[...])` → quita filas sin crecimiento. Al quitarlas, `pib_pc_usd` también queda sin NaN.

**Debe salir:** `(5172, 5)`.

# 11 · Comprobaciones

In [14]:
print(data_pib["iso3"].nunique())
print(data_pib.groupby(["iso3", "anio"]).size().max())
print(data_pib["anio"].min(), data_pib["anio"].max())
data_pib[(data_pib["iso3"] == "ESP") & (data_pib["anio"] >= 2023)]

209
1
2016 2024


,pais,iso3,anio,pib_pc_usd,crecim_pib_(%)
1542,Spain,ESP,2023,33493.222541,10.5
1543,Spain,ESP,2024,35326.768307,5.5


- **Debe salir:** `211` países, máximo `1` (sin duplicados), años `2001 2025` (2000 desaparece porque no tiene año anterior).
- España: 2023 → 10.47 · 2024 → 5.47 · 2025 → 9.34 (valores en %, dólares corrientes).

# 12 · Guardar

In [18]:
data_pib.to_csv("../data/depurados/limpio_pib.csv", index=False)